<a href="https://colab.research.google.com/github/hoapp07/PhuHoa_AI_UEH/blob/main/HTCApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

DATA_FILE = "dataxe.xlsx"
df_raw = pd.read_excel(DATA_FILE, engine='openpyxl')
df_raw.head()
print("Kích thước ban đầu:", df_raw.shape)


# Chuẩn hóa tên cột (bỏ khoảng trắng thừa)
df_raw.columns = [col.strip() for col in df_raw.columns]

# Đổi tên một số cột để dễ làm việc
rename_map = {
    'Số km đã chạy (ODO)': 'ODO_raw',
    'Tình trạng xe %': 'Tinh_trang_raw',
    'Giá bán': 'Gia_raw'
}
df_raw.rename(columns=rename_map, inplace=True)
# Hàm làm sạch số
def clean_number(x):
    """Chuyển chuỗi có dấu phẩy thành số"""
    if isinstance(x, str):
        x = x.replace(',', '').replace(' ', '')
    return pd.to_numeric(x, errors='coerce')

def clean_percent(x):
    """'99%' -> 99.0"""
    if isinstance(x, str):
        x = x.replace('%', '').strip()
    return pd.to_numeric(x, errors='coerce')

def clean_price(x):
    """'68,500,000 đ' -> giá trị tiền (VND), sau đó đổi ra triệu đồng"""
    if isinstance(x, str):
        x = x.lower().replace('đ', '').replace('vnd', '').replace(' ', '')
        x = x.replace(',', '')
    return pd.to_numeric(x, errors='coerce')

# Áp dụng
df_raw['So_km'] = df_raw['ODO_raw'].apply(clean_number)
df_raw['Tinh_trang'] = df_raw['Tinh_trang_raw'].apply(clean_percent)
df_raw['Gia_trieu'] = df_raw['Gia_raw'].apply(clean_price) / 1_000_000

# Năm sản xuất
df_raw['Nam_sx'] = pd.to_numeric(df_raw['Năm sản xuất'], errors='coerce')

# Chọn các cột cần thiết
cols = ['Hãng xe', 'Dòng xe', 'Nam_sx', 'So_km', 'Khu vực bán', 'Tinh_trang', 'Gia_trieu']
df = df_raw[cols].copy()

# Xóa dòng có giá trị rỗng
print("Số dòng trước dropna:", df.shape[0])
df = df.dropna()
print("Số dòng sau dropna:", df.shape[0])

# Xem qua dữ liệu
df.head()

print("Thông tin dữ liệu:")
print(df.info())
print("\nMô tả thống kê:")
print(df.describe(include='all'))

# Phân phối giá
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df['Gia_trieu'], kde=True, bins=30, ax=axes[0])
axes[0].set_title('Phân phối giá bán (triệu đồng)')
sns.boxplot(x=df['Gia_trieu'], ax=axes[1])
axes[1].set_title('Boxplot giá bán')
plt.tight_layout()
plt.show()

# Quan hệ giữa giá và các biến số
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(ax=axes[0], x=df['Nam_sx'], y=df['Gia_trieu'])
axes[0].set_title('Năm sản xuất vs Giá')
sns.scatterplot(ax=axes[1], x=df['So_km'], y=df['Gia_trieu'])
axes[1].set_title('Số km vs Giá')
sns.scatterplot(ax=axes[2], x=df['Tinh_trang'], y=df['Gia_trieu'])
axes[2].set_title('Tình trạng vs Giá')
plt.tight_layout()
plt.show()

# Giá trung bình theo hãng
plt.figure(figsize=(10, 5))
df.groupby('Hãng xe')['Gia_trieu'].mean().sort_values().plot(kind='barh')
plt.title('Giá trung bình theo Hãng xe')
plt.xlabel('Triệu đồng')
plt.show()

# Tạo tuổi xe
df['Tuoi_xe'] = 2026 - df['Nam_sx']
df.drop('Nam_sx', axis=1, inplace=True)

# Chuẩn hóa tên dòng xe (sửa lỗi chính tả phổ biến)
corrections = {
    'Air Blade': 'Air Blade',
    'Air blade': 'Air Blade',
    'AirBlade': 'Air Blade',
    'Airblade': 'Air Blade',
    'Vario': 'Vario',
    'Valro': 'Vario',
    'Vairo': 'Vario',
    'LEAD': 'Lead',
    'lead': 'Lead',
    'Vision': 'Vision',
    'Visson': 'Vision',
    'SH Mode': 'SH Mode',
    'SH': 'SH',
    'Winner X': 'Winner',
    'Winner V1': 'Winner',
    'Exciter': 'Exciter',
    'Excite': 'Exciter'
}
df['Dòng xe'] = df['Dòng xe'].replace(corrections)

# Loại bỏ outliers giá (theo IQR)
Q1 = df['Gia_trieu'].quantile(0.25)
Q3 = df['Gia_trieu'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df = df[(df['Gia_trieu'] >= lower) & (df['Gia_trieu'] <= upper)]
print(f"Số dòng sau khi lọc outliers giá: {df.shape[0]}")

# Xáo trộn dữ liệu
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Tách X và y
X = df.drop('Gia_trieu', axis=1)
y = df['Gia_trieu']

# Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Kích thước train: {X_train.shape}, test: {X_test.shape}")
# Xác định danh sách cột theo kiểu dữ liệu
numeric_features = ['So_km', 'Tinh_trang', 'Tuoi_xe']
categorical_features = ['Hãng xe', 'Dòng xe', 'Khu vực bán', 'Đã thay phụ tùng chưa?']
# Lọc những cột thực sự tồn tại
categorical_features = [c for c in categorical_features if c in X.columns]
numeric_features = [c for c in numeric_features if c in X.columns]
print("Cột số:", numeric_features)
print("Cột phân loại:", categorical_features)
#XÂY DỰNG PIPELINE & HUẤN LUYỆN CÁC MÔ HÌNH
preprocessor_linear = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
# Preprocessor cho cây (không cần scale)
preprocessor_tree = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
# -------- Linear Regression --------
pipe_lr = Pipeline([
    ('prep', preprocessor_linear),
    ('reg', LinearRegression())
])
pipe_lr.fit(X_train, y_train)

# -------- Ridge Regression --------
pipe_ridge = Pipeline([
    ('prep', preprocessor_linear),
    ('reg', Ridge())
])
param_ridge = {'reg__alpha': [0.1, 1, 10, 100]}
grid_ridge = GridSearchCV(pipe_ridge, param_ridge, cv=5,
                          scoring='neg_root_mean_squared_error')
grid_ridge.fit(X_train, y_train)
best_ridge = grid_ridge.best_estimator_
print("Ridge best alpha:", grid_ridge.best_params_)

# -------- Decision Tree --------
pipe_dt = Pipeline([
    ('prep', preprocessor_tree),
    ('reg', DecisionTreeRegressor(random_state=42))
])
param_dt = {
    'reg__max_depth': [5, 10, 15, None],
    'reg__min_samples_split': [2, 5, 10],
    'reg__min_samples_leaf': [1, 2, 4]
}
grid_dt = GridSearchCV(pipe_dt, param_dt, cv=5,
                       scoring='neg_root_mean_squared_error')
grid_dt.fit(X_train, y_train)
best_dt = grid_dt.best_estimator_
print("Decision Tree best params:", grid_dt.best_params_)
# -------- Random Forest --------
pipe_rf = Pipeline([
    ('prep', preprocessor_tree),
    ('reg', RandomForestRegressor(random_state=42, n_jobs=-1))
])
param_rf = {
    'reg__n_estimators': [100, 200],
    'reg__max_depth': [10, 15, None],
    'reg__min_samples_split': [2, 5],
    'reg__min_samples_leaf': [1, 2]
}
grid_rf = GridSearchCV(pipe_rf, param_rf, cv=5,
                       scoring='neg_root_mean_squared_error')
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
print("Random Forest best params:", grid_rf.best_params_)

# ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST

def evaluate_model(name, model):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"\n{'='*40}")
    print(f"{name}")
    print(f"  MAE  : {mae:.2f} triệu đồng")
    print(f"  RMSE : {rmse:.2f} triệu đồng")
    print(f"  R²   : {r2:.4f}")
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

results = []
results.append(evaluate_model("Linear Regression", pipe_lr))
results.append(evaluate_model("Ridge Regression", best_ridge))
results.append(evaluate_model("Decision Tree (tuned)", best_dt))
results.append(evaluate_model("Random Forest (tuned)", best_rf))

# Bảng so sánh
df_results = pd.DataFrame(results)
print("\n===== BẢNG SO SÁNH MÔ HÌNH =====")
print(df_results.to_string(index=False))

# Chọn mô hình tốt nhất (RMSE nhỏ nhất)
best_rmse = float('inf')
best_model = None
best_model_name = ""
model_map = {
    'Linear Regression': pipe_lr,
    'Ridge Regression': best_ridge,
    'Decision Tree (tuned)': best_dt,
    'Random Forest (tuned)': best_rf
}
for name, model in model_map.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_model_name = name




st.set_page_config(page_title="Dự đoán giá xe máy cũ", page_icon="🛵")
st.title("🛵 Dự đoán giá bán xe máy cũ (TP.HCM)")
st.markdown("Nhập thông tin xe để nhận giá dự đoán (triệu đồng)")

# Layout 2 cột
col1, col2 = st.columns(2)
with col1:
    hang = st.selectbox("Hãng xe", ["Honda", "Yamaha", "SYM", "Piaggio", "Suzuki", "Detech", "MOKA", "Khác"])
    dong_xe = st.text_input("Dòng xe (VD: Wave, SH, Exciter...)", "Wave")
    nam_sx = st.number_input("Năm sản xuất", min_value=1990, max_value=2026, value=2020)
    so_km = st.number_input("Số km đã chạy", min_value=0, max_value=500000, value=15000)
with col2:
    tinh_trang = st.slider("Tình trạng xe (1-10)", 1, 10, 9)
    thay_phutung = st.radio("Đã thay phụ tùng chưa?", ["No", "Yes"])
    khu_vuc = st.text_input("Khu vực bán (VD: Quận 7, Hồ Chí Minh)", "Quận 7, Hồ Chí Minh")

if st.button("🔍 Dự đoán giá"):
    # Chuẩn bị dữ liệu input
    input_data = {
        "Hãng xe": hang,
        "Dòng xe": dong_xe,
        "So_km": so_km,
        "Tinh_trang": tinh_trang * 10,   # thang 1-10 -> % (10-100)
        "Tuoi_xe": 2026 - nam_sx,
        "Khu vực bán": khu_vuc,
        "Đã thay phụ tùng chưa?": thay_phutung
    }
    input_df = pd.DataFrame([input_data])
    # Sắp xếp cột theo đúng feature_names_in_ của pipeline
    input_df = input_df[model.feature_names_in_]
    # Dự đoán
    gia = model.predict(input_df)[0]
    st.success(f"💰 Giá bán dự kiến: **{gia:,.2f} triệu đồng**")
    st.info("Lưu ý: Đây là giá tham khảo, có thể thay đổi theo tình trạng thực tế và thị trường.")

Writing app.py


In [1]:
!pip install streamlit pandas joblib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 106.2 MB/s eta 0:00:00
